In [1]:
import requests
from pathlib import Path
import time

def download_iiif_book(manifest_url, output_dir, delay=2.0):
    """Polite IIIF book downloader. delay = seconds between requests."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    manifest = requests.get(manifest_url).json()
    
    # IIIF v2: sequences[0].canvases ; v3: items
    canvases = manifest.get('sequences', [{}])[0].get('canvases') \
               or manifest.get('items', [])
    
    for i, canvas in enumerate(canvases):
        # Extract image service URL
        img_service = (canvas.get('images', [{}])[0]
                            .get('resource', {})
                            .get('service', {})
                            .get('@id')
                       or canvas.get('items', [{}])[0]
                            .get('items', [{}])[0]
                            .get('body', {})
                            .get('service', [{}])[0]
                            .get('id'))
        
        # Request maximum quality
        url = f"{img_service}/full/max/0/default.jpg"
        out_path = Path(output_dir) / f"page_{i+1:04d}.jpg"
        
        if out_path.exists():
            continue
        
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        out_path.write_bytes(r.content)
        print(f"✓ Page {i+1}/{len(canvases)}")
        time.sleep(delay)  # be polite to the server